In [0]:
import dlt
from pyspark.sql.functions import col, count, max as spark_max

In [0]:
@dlt.table(
    comment="Asteroid approaches from NeoWs API (silver layer)"
)
def neows_asteroids():
    return spark.read.table("nasa_analytics.silver.neows_asteroids")

@dlt.table(
    comment="DONKI notifications (silver layer)"
)
def donki_notifications():
    return spark.read.table("nasa_analytics.silver.donki_notifications")

In [0]:
@dlt.table(
    comment="Daily hazardous asteroid metrics"
)
def neows_hazard_summary():
    return (
        dlt.read("neows_asteroids")
        .groupBy("approach_date")
        .agg(
            count("*").alias("total_asteroids"),
            count(when(col("is_hazardous") == True, 1)).alias("hazardous_count"),
            spark_max("velocity_kph").alias("max_velocity_kph")
        )
    )

In [0]:
@dlt.table(
    comment="DONKI event counts by day"
)
def donki_event_summary():
    return (
        dlt.read("donki_notifications")
        .groupBy(col("issue_time").cast("date").alias("event_date"), col("message_type"))
        .agg(count("*").alias("event_count"))
    )

In [0]:
@dlt.table(
    comment="Unified space hazard dashboard combining asteroid and solar activity"
)
def space_hazard_dashboard():
    neows = dlt.read("neows_hazard_summary")
    donki = dlt.read("donki_event_summary")
    
    return (
        neows.join(
            donki,
            neows.approach_date == donki.event_date,
            "outer"
        )
        .select(
            col("approach_date").alias("date"),
            "total_asteroids",
            "hazardous_count",
            "max_velocity_kph",
            "message_type",
            "event_count"
        )
    )